# Your first PIEC experiment — local testing
Please download this file and run locally not within the repository root to avoid tracking changes.
Now try sine, square, and ramp waves virtually or on your bench.

**Run the imports cell below first**, then choose where to start:

- **No hardware:** skip to [A. Virtual instruments](#A.-Virtual-instruments).
- **Real hardware:** skip to [B. Physical instruments](#B.-Physical-instruments).

Run **one cell at a time with Shift+Enter** within your chosen section. Each section works independently.

From the repository root, install with `python -m pip install -e . jupyterlab` and launch `python -m jupyter lab`. Physical instruments also need a working VISA backend; see the [connection guide](https://piec.readthedocs.io/en/latest/user_guide/connecting_to_instrument.html).

Prefer a GUI? From the repository root, run `python Measurements/Quickstart/awg_scope_GUI.py`. It provides the same virtual and Keysight physical examples with settings, plots, and CSV export.

In [ ]:
#Our imports for the entire notebook
import numpy as np
import matplotlib.pyplot as plt

from piec.drivers.awg.virtual_awg import VirtualAwg
from piec.drivers.oscilloscope.virtual_oscilloscope import VirtualScope
from piec.drivers.awg.k_81150a import Keysight81150a
from piec.drivers.oscilloscope.k_dsox3024a import KeysightDSOX3024a

## A. Virtual instruments

Start here without hardware. The first plot shows the applied waveform; the second shows the existing virtual ferroelectric sample response.

In [ ]:
awg = VirtualAwg(simulation_points=1000)
scope = VirtualScope()
print(awg.idn(), "and", scope.idn())

### 2. Start with a sine wave

Set channel 1 to a 1 kHz sine wave. The amplitude is 1 V peak in this simulation.

In [ ]:
frequency_hz = 1_000
amplitude_v = 1.0
#pro tip, hover over the arguments to see what they do, or check the docs for more details
awg.configure_waveform(
    1, "SIN", frequency=frequency_hz,
    amplitude=amplitude_v, offset=0.0,
)

Run the next cell to send the sine wave and see the result.

In [ ]:
applied_voltage = awg.get_waveform(1)
applied_time = np.linspace(0, 1 / frequency_hz, len(applied_voltage))

try:
    awg.output(1, on=True)
    scope.arm()
    awg.send_software_trigger()
    sine_data = scope.get_data()
finally:
    awg.output(1, on=False)

fig, axes = plt.subplots(2, 1, figsize=(8, 5))
axes[0].plot(applied_time * 1_000, applied_voltage)
axes[0].set(title=f"Applied SIN: {frequency_hz:g} Hz",
            xlabel="Time (ms)", ylabel="Applied voltage (V)")
axes[1].plot(sine_data["Time"] * 1_000, sine_data["Voltage"])
axes[1].set(title="Ferroelectric sample response (50 Ω sensing resistor)",
            xlabel="Time (ms)", ylabel="Scope voltage (V)")
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
sine_data.head()

### 3. Try a square wave

Now change only the waveform. The frequency and amplitude stay the same.

In [ ]:
#pro tip, piec is built to be case-insensitive, so 'squ' will also work
awg.set_waveform(1, "SQU")

Run the next cell to send the square wave and see the result.

In [ ]:
applied_voltage = awg.get_waveform(1)
applied_time = np.linspace(0, 1 / frequency_hz, len(applied_voltage))

try:
    awg.output(1, on=True)
    scope.arm()
    awg.send_software_trigger()
    square_data = scope.get_data()
finally:
    awg.output(1, on=False)

fig, axes = plt.subplots(2, 1, figsize=(8, 5))
axes[0].plot(applied_time * 1_000, applied_voltage)
axes[0].set(title=f"Applied SQU: {frequency_hz:g} Hz",
            xlabel="Time (ms)", ylabel="Applied voltage (V)")
axes[1].plot(square_data["Time"] * 1_000, square_data["Voltage"])
axes[1].set(title="Ferroelectric sample response (50 Ω sensing resistor)",
            xlabel="Time (ms)", ylabel="Scope voltage (V)")
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
square_data.head()

### 4. Try a ramp

Now change only the waveform. The frequency and amplitude stay the same.

In [ ]:
awg.set_waveform(1, "RAMP")

Run the next cell to send the ramp and see the result.

In [ ]:
applied_voltage = awg.get_waveform(1)
applied_time = np.linspace(0, 1 / frequency_hz, len(applied_voltage))

try:
    awg.output(1, on=True)
    scope.arm()
    awg.send_software_trigger()
    ramp_data = scope.get_data()
finally:
    awg.output(1, on=False)

fig, axes = plt.subplots(2, 1, figsize=(8, 5))
axes[0].plot(applied_time * 1_000, applied_voltage)
axes[0].set(title=f"Applied RAMP: {frequency_hz:g} Hz",
            xlabel="Time (ms)", ylabel="Applied voltage (V)")
axes[1].plot(ramp_data["Time"] * 1_000, ramp_data["Voltage"])
axes[1].set(title="Ferroelectric sample response (50 Ω sensing resistor)",
            xlabel="Time (ms)", ylabel="Scope voltage (V)")
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()
ramp_data.head()

### 5. Save your results

Save each response as a CSV. Rerunning this cell replaces these CSVs.

In [ ]:
sine_data.to_csv("sine.csv", index=False)
square_data.to_csv("square.csv", index=False)
ramp_data.to_csv("ramp.csv", index=False)

**Virtual section complete.** Stop here, or continue below with connected instruments.

For the bench comparison, compare the physical trace with the **applied waveform** above. The virtual response includes a ferroelectric sample; a direct AWG-to-scope connection does not.

## B. Physical instruments

Start here after the imports cell. This example uses a **Keysight 81150A AWG** and **Keysight DSOX3024A scope**. For other models, adapt the drivers and acquisition settings.

1. Connect **AWG channel 1 output → scope channel 1 input** with a BNC cable.
2. On the AWG, select continuous output (burst off) and amplitude units of Vpp.
3. The cells below set both loads to **50 Ω** and trigger the scope on **channel 1, rising edge, 0 V**. No separate trigger cable is needed.

If you prefer external triggering, connect **AWG trigger/sync out → scope external trigger input**, then change the scope trigger source to `"EXT"` and choose a level appropriate for that signal.

Enter your VISA addresses below. The hardware section has not been validated on a physical bench.

In [ ]:
awg_address = "YOUR_AWG_VISA_ADDRESS"
scope_address = "YOUR_SCOPE_VISA_ADDRESS"

if awg_address.startswith("YOUR_") or scope_address.startswith("YOUR_"):
    raise ValueError("Enter both instrument VISA addresses before running this cell.")

physical_awg = Keysight81150a(awg_address)
physical_awg.output(1, on=False)
physical_scope = KeysightDSOX3024a(scope_address)
physical_scope.instrument.timeout = 10_000  # milliseconds
print(physical_awg.idn())
print(physical_scope.idn())

### Set up the scope

Use a 5 ms window to see five cycles at 1 kHz. Each capture below switches the AWG output off when finished.

In [ ]:
physical_frequency_hz = 1_000
physical_amplitude_vpp = 2.0  # Matches the virtual example's 1 V peak

physical_awg.set_source_impedance(1, 50)
physical_awg.set_load_impedance(1, 50)
physical_awg.set_trigger_source(1, "IMM")
physical_scope.toggle_channel(1, on=True)
physical_scope.set_channel_impedance(1, "50")
physical_scope.set_input_coupling(1, "DC")
physical_scope.set_probe_attenuation(1, 1)
physical_scope.set_vertical_scale(1, vdiv=0.5)
physical_scope.set_vertical_position(1, 0)
physical_scope.configure_horizontal(x_range=0.005, x_position=0)
physical_scope.configure_trigger(
    trigger_source=1, trigger_mode="EDGE", trigger_slope="POS", trigger_level=0,
)
physical_scope.set_trigger_sweep("NORM")
physical_scope.configure_acquisition(
    channel=1, acquisition_mode="NORM", acquisition_points=1000,
)
physical_scope.instrument.write(":WAVeform:FORMat BYTE")
physical_scope.instrument.write(":WAVeform:UNSigned OFF")

### Try a sine wave

Configure the signal, then run the next cell to capture it.

In [ ]:
physical_awg.configure_waveform(
    1, "SIN", frequency=physical_frequency_hz,
    amplitude=physical_amplitude_vpp, offset=0,
)

In [ ]:
try:
    physical_awg.output(1, on=True)
    physical_awg.operation_complete()
    physical_scope.set_acquisition()
    physical_scope.operation_complete()
    physical_sine_data = physical_scope.get_data()
finally:
    physical_awg.output(1, on=False)

plt.figure(figsize=(8, 3))
plt.plot(physical_sine_data["Time"] * 1_000, physical_sine_data["Voltage"])
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (V)")
plt.title("Physical sine wave")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Try a square wave

Configure the signal, then run the next cell to capture it.

In [ ]:
physical_awg.set_waveform(1, "SQU")
physical_awg.set_square_duty_cycle(1, 50)

In [ ]:
try:
    physical_awg.output(1, on=True)
    physical_awg.operation_complete()
    physical_scope.set_acquisition()
    physical_scope.operation_complete()
    physical_square_data = physical_scope.get_data()
finally:
    physical_awg.output(1, on=False)

plt.figure(figsize=(8, 3))
plt.plot(physical_square_data["Time"] * 1_000, physical_square_data["Voltage"])
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (V)")
plt.title("Physical square wave")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Try a ramp wave

Configure the signal, then run the next cell to capture it.

In [ ]:
physical_awg.set_waveform(1, "RAMP")
physical_awg.set_ramp_symmetry(1, 100)  # Rising sawtooth, as in the virtual example

In [ ]:
try:
    physical_awg.output(1, on=True)
    physical_awg.operation_complete()
    physical_scope.set_acquisition()
    physical_scope.operation_complete()
    physical_ramp_data = physical_scope.get_data()
finally:
    physical_awg.output(1, on=False)

plt.figure(figsize=(8, 3))
plt.plot(physical_ramp_data["Time"] * 1_000, physical_ramp_data["Voltage"])
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (V)")
plt.title("Physical ramp wave")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Save and disconnect

These filenames keep the physical captures separate from the virtual results. Rerunning replaces the files.

In [ ]:
physical_sine_data.to_csv("physical_sine.csv", index=False)
physical_square_data.to_csv("physical_square.csv", index=False)
physical_ramp_data.to_csv("physical_ramp.csv", index=False)

physical_awg.output(1, on=False)
physical_awg.instrument.close()
physical_scope.instrument.close()